# Lab 11 · Keep the reads that match

**Today:** when you walk out, chapter 4's filtering computation, simulate, mask and count, is something you can write from a blank cell.

**Before you start:** chapter 4 read, and lab 10's masks. Today they do the chapter's work on the chapter's own reads.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. Every task has a hint in the **Hints**
block at the end of the notebook, for when you want it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## 1 · A label on every simulated read

Chapter 4 asks which of two organisms produced a 40-base read with 25 GC bases. Its model makes a read in two steps: one organism is selected, Microbe A with probability 0.5 and Microbe B otherwise, then a GC count is drawn as a binomial at that organism's rate, 0.66 for A and 0.48 for B. The cell below is the chapter's simulator and its hundred-thousand-read loop, unchanged. `draw_gc_from_two_groups` returns two values, `species, gc_count`, and the call `s, g = draw_gc_from_two_groups(...)` puts them into two names at once. Returning a pair is new in this unit, and reading it is all this lab asks of it.

The loop ends with two arrays of the same length, positions aligned: `species[i]` is the organism that produced read `i`, and `gc[i]` is its GC count. The collaborator's real read has a GC count and no label. Every simulated read has both, because the simulator wrote it, and the alignment is what lets a mask built on one array keep entries of the other. **Predict before you run: the chapter prints B A B B A and 22 24 18 20 26 as its first five reads at seed 41. Will this cell print the same five, and what share of the reads will be labeled A?**

In [ ]:
import numpy as np

# the seeded generator every draw in this section uses
rng = np.random.default_rng(41)

def draw_gc_from_two_groups(share_a, gc_a, gc_b, n_bases, rng):
    """One read from the mixed community: which organism produced it,
    then how many of its bases are G or C."""
    # randomly choose A or B
    if rng.random() < share_a:
        species = "A"
        # if we chose A, sample a GC content centered at gc_a
        gc_count = rng.binomial(n_bases, gc_a)
    else:
        species = "B"
        # if we chose B, sample a GC content centered at gc_b
        gc_count = rng.binomial(n_bases, gc_b)
    # return both the selected species and the GC count
    return species, gc_count

rng = np.random.default_rng(41)
species_list, gc_list = [], []
n_reads = 100000
for _ in range(n_reads):
    # draw a species and a GC count from A or B with equal proportions
    s, g = draw_gc_from_two_groups(0.5, 0.66, 0.48, 40, rng)
    # record the species and GC count
    species_list.append(s)
    gc_list.append(g)
species = np.array(species_list)
gc = np.array(gc_list)
print("first five reads:")
print(f"species: {species[:5]}, GC count: {gc[:5]}")
is_a = (species == "A")   # a mask: True for each read that came from A
print("share of reads labeled A:", round(is_a.mean(), 3))

The same five, because the seed is the same and every draw happens in the same order. The share is about 0.5, as built: the chapter reads it off the labels as `is_a.mean()`, the way lab 10 counted with a mask.

**Test your understanding.** Create two variables from the two arrays above, using masks and no loops. `a_mean` stores one number, the average GC count of the reads labeled A, rounded to 1 decimal. `n_gc_rich_b` stores one whole number, how many reads labeled *B* still have more than 25 GC bases. This is question 1. Its hint is at the end of the notebook.

In [ ]:
# your turn: two variables named a_mean and n_gc_rich_b (use the species and gc arrays above)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("a_mean", expect=26.4,
      hint="mask from species, filter gc, .mean(), round 1")
check("n_gc_rich_b", expect=1060,
      hint="two masks combined with & — B AND above 25")

## 2 · The filtering answer

The collaborator's read has **exactly 25 GC bases**, and the sequencer does not record which organism wrote it. Chapter 4's filtering computation, construct by construct: mask the simulated reads to the ones that look like this one, then ask what fraction of *those* came from A. The cell is the chapter's chunk, unchanged. **Predict roughly, then run: A's reads average 26.4 GC bases and B's 19.2, so 25 sits between the two averages. Will the answer be near 0, near 1, or genuinely in between?**

In [ ]:
observed = 25

# the mask: which simulated reads match hers
looks_like_observed = (gc == observed)
# how many Trues
n_matching = looks_like_observed.sum()
# how many of the simulated GC = 25 reads came from each organism
from_a = (species[looks_like_observed] == "A")
from_b = (species[looks_like_observed] == "B")
print(f"reads that look like hers: {n_matching}")
print(f"fraction of those from A:  {from_a.mean():.2f}")
print(f"fraction of those from B:  {from_b.mean():.2f}")

Genuinely in between, and nearer 1: 25 is reachable by a GC-rich B read or an ordinary A read, and the split of the matching reads between A and B is the answer. That share is a probability, computed by counting. Chapter 4 calls it the **posterior**: a probability worked out after looking at the data, here the count of 25. About seven thousand reads match, and 0.83 of them came from A.

**Test your understanding.** Write a function named `share_a_given` that takes three parameters: `count`, a whole number of GC bases; `species`, the array of labels; and `gc`, the array of GC counts. It should return one number: the fraction of the simulated reads with exactly `count` GC bases that came from A, rounded to 2 decimals. Then create two variables by calling it on the arrays above: `share_at_21`, the share at 21 GC bases, and `share_at_23`, the share at 23. Predict the order before running the check: 21 sits nearer B's average of 19.2, and 23 sits almost exactly halfway between the two averages. This is question 2. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named share_a_given(count, species, gc), then two variables named share_at_21 and share_at_23

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("share_at_21", expect=0.21,
      hint="mask on gc == 21, then the share from A among the kept, rounded to 2")
check("share_at_23", expect=0.53,
      hint="the crossover: halfway between the two averages, the organisms split the matching reads almost evenly")

## If you finish early

- Ask section 2 about 20 and 28 GC bases, and watch the answer run from near 0 to near 1.
- Rerun section 1 with A's share at 0.05 instead of 0.5, as chapter 4 does under *Playing with the story*, and predict which way 0.83 moves before you run.
- Chapter 4's practice problem 4.1 asks the same question of the chapter's community with the mixing proportion changed.

## If you are stuck

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`a_mean` is close but not exact.** The mask must come from `species` and the mean from `gc[is_a]`; rounding to 1 decimal happens last.
- **`share_a_given` errors on empty matches.** At counts nobody simulated, the filtered array is empty and `.mean()` warns. For today's checks the counts always match some reads; the empty case is chapter 4's pseudocount discussion, worth rereading.
- **Everything passes but slowly.** A mask does 100,000 comparisons per line, which is normal and fast. The loop that would have done the same work is the one you did not have to write.

## Hints

**Question 1 · `a_mean`, `n_gc_rich_b`.** Build a mask on `species` first: `is_a = (species == "A")`, which section 1 already did. Then `gc[is_a].mean()`, rounded to 1 decimal, is `a_mean`, the way lab 10's section 2 filtered one array with a mask built from another. For `n_gc_rich_b`, combine two masks with `&`, as in lab 10's section 3: `(species == "B") & (gc > 25)`, each comparison in its own parentheses, then `.sum()`.

**Question 2 · `share_a_given`.** Section 2's chunk is the function body with 25 replaced by `count`. Inside the function, build `looks_like = (gc == count)`, then return `round((species[looks_like] == "A").mean(), 2)`. Then two plain assignment lines call it: `share_at_21 = share_a_given(21, species, gc)`, and the same at 23.